# Run17 Expanded Universe Training (Local)
Thin orchestration notebook for the canonical Run17 expanded-universe training path.
20-asset universe with 10-year train (2010-2019) / 5-year test (2020-2024) split.
Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/environment source files.

## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [ ]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_BRANCH = None  # e.g. "feature/run10-alpha-overhaul"
INSTALL_REQUIREMENTS = False
RESET_OUTPUT_DIRS = True

TRAIN_REPO_CANDIDATES = [
    Path.cwd(),
    Path(r"C:\Users\Owner\tcn_tape_vectorized_version_clean"),
    Path("/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean"),
]
TRAIN_REPO_DIR = next((p for p in TRAIN_REPO_CANDIDATES if (p / ".git").exists()), None)

def run(cmd):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

if TRAIN_REPO_DIR is None:
    attempted = "\n".join(f" - {p}" for p in TRAIN_REPO_CANDIDATES)
    raise FileNotFoundError("Local repo not found. Tried:\n" + attempted)

if TRAIN_BRANCH:
    run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"])
    run(["git", "-C", str(TRAIN_REPO_DIR), "checkout", TRAIN_BRANCH])

if RESET_OUTPUT_DIRS:
    purge_paths = [
        TRAIN_REPO_DIR / "tcn_fusion_results",
        TRAIN_REPO_DIR / "tcn_results",
        TRAIN_REPO_DIR / "tcn_att_results",
        TRAIN_REPO_DIR / "output_log",
        TRAIN_REPO_DIR / "output_logs",
        TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
        TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
        TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",
        TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",
    ]
    for path in purge_paths:
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        elif path.exists():
            path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

if INSTALL_REQUIREMENTS:
    requirements_file = TRAIN_REPO_DIR / "requirements.txt"
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])

print("[OK] Local repo ready:", TRAIN_REPO_DIR)
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "HEAD"])
print("[OK] Requirements installed:", INSTALL_REQUIREMENTS)


In [ ]:
import tensorflow as tf
import subprocess

REQUIRE_GPU = False  # set True if you want hard fail without GPU

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if smi.returncode == 0:
        print("nvidia-smi:", [line.strip() for line in smi.stdout.splitlines() if line.strip()])
    else:
        print("nvidia-smi: not available")
except Exception:
    print("nvidia-smi: not available")

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)

if not gpus:
    msg = 'No GPU visible to TensorFlow; running on CPU (slower).'
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('[WARN]', msg)
else:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('TF build CUDA:', tf.test.is_built_with_cuda())


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run17_expanded_config, assert_run17_expanded_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import (
    _get_results_root_for_architecture,
    prepare_phase1_dataset,
    run_experiment6_tape,
)

RUN_ID = 'run17'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None


In [ ]:
import logging

QUIET_SRC_INFO_LOGS = True

if QUIET_SRC_INFO_LOGS:
    # Suppress noisy module INFO logs like: "... - src.environment_tape_rl - INFO ..."
    for logger_name in [
        "src",
        "src.environment_tape_rl",
        "src.notebook_helpers.tcn_phase1",
        "src.agents.ppo_agent_tf",
    ]:
        logging.getLogger(logger_name).setLevel(logging.WARNING)
    print("[OK] Suppressed src INFO logs (level=WARNING).")
else:
    print("[INFO] Src INFO logs left enabled.")


## 3) Build Canonical Run17 Config and Dataset
Create the source-backed Run17 config (20-asset expanded universe), assert no drift, and prepare the dataset once.

In [ ]:
train_config = build_run17_expanded_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run17_expanded_config(train_config)

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']

print('[Run17] Canonical config ready')
print('  num_assets =', train_config['NUM_ASSETS'])
print('  tickers =', train_config['ASSET_TICKERS'])
print('  analysis_start_date =', train_config['ANALYSIS_START_DATE'])
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  regime_conditioning =', ap['regime_conditioning_enabled'])
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  cvar_advantage_weight =', ppo['cvar_advantage_weight'])
print('  lagrangian =', {
    'threshold': ppo['lagrangian_cvar_threshold'],
    'lr': ppo['lagrangian_cvar_lr'],
    'lambda_max': ppo['lagrangian_cvar_lambda_max'],
    'penalty_scale': ppo['lagrangian_cvar_penalty_scale'],
})
print('  drawdown =', {
    'target': env['drawdown_constraint']['target'],
    'tolerance': env['drawdown_constraint']['tolerance'],
    'penalty_coef': env['drawdown_constraint']['penalty_coef'],
    'lambda_carry_decay': env['drawdown_constraint']['lambda_carry_decay'],
})
print('  dispersion =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'dispersion_target_std': ppo['alpha_dispersion_target_std'],
})
print('  deterministic_validation =', tp['deterministic_validation_checkpointing_enabled'])
print('  training_early_stop =', tp.get('training_early_stop_enabled', False))

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
if actuarial_cols:
    raise RuntimeError(f'Actuarial columns should be absent for Run17: {actuarial_cols}')

alpha_ret_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('AlphaRet_')]
expected_alpha_cols = {'AlphaRet_1d', 'AlphaRet_5d', 'AlphaRet_20d', 'AlphaRet_5d_Z', 'AlphaRet_20d_Z'}
missing_alpha_cols = sorted(list(expected_alpha_cols - set(alpha_ret_cols)))
if missing_alpha_cols:
    raise RuntimeError(f'Missing expected alpha-return columns: {missing_alpha_cols}')

fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Actuarial feature check passed: none present (disabled)')
print('[OK] Alpha-return feature check passed:', sorted(alpha_ret_cols)[:10])
print('[OK] Fundamental feature check passed: none present')

## 4) Run Training
Launch the canonical Run17 training path.


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [ ]:
TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
    architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
    use_attention=bool(train_config['agent_params'].get('use_attention', False)),
    use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
    project_root=TRAIN_REPO_DIR,
)
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

print('Results root:', TRAIN_RESULTS_ROOT)
print('Logs dir:', TRAIN_LOGS_DIR)

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


## 6) Export Artifacts (Optional)
Zip the latest results and save them into a dedicated Google Drive folder for this run.


In [ ]:
import shutil
import subprocess

EXPORT_RESULTS_ZIP = False
SAVE_TO_DRIVE = True
DRIVE_EXPORT_ROOT = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs')
DRIVE_EXPORT_DIR = DRIVE_EXPORT_ROOT / RUN_ID
LOCAL_EXPORT_PATH = TRAIN_REPO_DIR / f"tcn_tape_vectorized_{RUN_ID}.zip"

if 'TRAIN_RESULTS_ROOT' not in globals():
    TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
        architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
        use_attention=bool(train_config['agent_params'].get('use_attention', False)),
        use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
        project_root=TRAIN_REPO_DIR,
    )

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_RESULTS_ROOT,
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if LOCAL_EXPORT_PATH.exists():
            LOCAL_EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            [
                'bash',
                '-lc',
                'cd "{}" && zip -qr "{}" {}'.format(
                    TRAIN_REPO_DIR,
                    LOCAL_EXPORT_PATH,
                    ' '.join(f'"{item}"' for item in relative_items),
                ),
            ],
            check=True,
        )
        print('[OK] Created local zip:', LOCAL_EXPORT_PATH)

        if SAVE_TO_DRIVE:
            if not Path('/content/drive/MyDrive').exists():
                raise FileNotFoundError('Google Drive is not mounted at /content/drive/MyDrive')
            DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
            drive_zip_path = DRIVE_EXPORT_DIR / LOCAL_EXPORT_PATH.name
            shutil.copy2(LOCAL_EXPORT_PATH, drive_zip_path)
            print('[OK] Copied zip to Drive:', drive_zip_path)
            print('[OK] Drive run folder:', DRIVE_EXPORT_DIR)
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')
    print('Drive export folder would be:', DRIVE_EXPORT_DIR)
